# Training 

In [1]:
import gymnasium
from stable_baselines3 import PPO, DDPG, SAC, A2C, TD3
from stable_baselines3.common.callbacks import EvalCallback, StopTrainingOnNoModelImprovement
import os
import utils
from gymnasium.envs.registration import register
import config
import shutil

In [2]:
register(
    id="reactor_v2",
    entry_point="reactor_env:Reactor",
    kwargs={'experiment_name':"default"}
)

In [3]:
# Experiment Details
experiment_name = config.EXPERIMENT_NAME
# Model Specifications
model_name = config.MODEL

# Logfiles and model save path
trained_model_name = 'sac_cn10cb100_1'
trained_model_path = os.path.join('experiments', trained_model_name, 'model', 'best_model.zip')
models_dir = f'experiments/{experiment_name}/model'
logdir = f'logs'

In [4]:
# Create paths if they don't exist
if not os.path.exists(f"experiments/{experiment_name}"):
    os.makedirs(f"experiments/{experiment_name}")

if not os.path.exists(models_dir):
    os.makedirs(models_dir)

if not os.path.exists(logdir):
    os.makedirs(logdir)

In [5]:
# Copy files to experiment folder
files = os.listdir("copy_scripts")
utils.copy_files(files)
shutil.copy('config.py', f"experiments/{config.EXPERIMENT_NAME}")
shutil.copy('utils.py', f"experiments/{config.EXPERIMENT_NAME}")

'experiments/sac_cn10cb100_1_ret\\utils.py'

In [6]:
# Compile the environment
env = gymnasium.make('reactor_v2', experiment_name=experiment_name)
env.reset()

(array([0.  , 0.  , 0.67]), {})

In [7]:
model = SAC.load(trained_model_path, env=env, device='cuda')  # Load the pretrained model

In [8]:
from episode_count_callback import EpisodeCounterCallback
# Initialize an empty list to store episode counts
episode_counts = []

# Create the callback and pass the list to it
episode_counter = EpisodeCounterCallback(episode_list=episode_counts)

In [9]:
stop_train_callback = StopTrainingOnNoModelImprovement(max_no_improvement_evals=1, min_evals=5, verbose=1)

In [10]:
# Evaluation Callback
eval_callback = EvalCallback(
    env, 
    best_model_save_path=models_dir,
    n_eval_episodes=50,
    eval_freq=10_000,
    verbose=1,
    deterministic= False,
    callback_after_eval= stop_train_callback
)

In [11]:
# Training loop
TIMESTEPS = 5e6
EPOCHS = 1
model_save_path = os.path.join(models_dir,f'{experiment_name}.zip')
print(f"-------------------- Running Experiment: {experiment_name} -------------------- ")
print(f"-------------------- TRAINING {model_name} --------------------")
for i in range(1, EPOCHS + 1):
    print(f"Training {i}/{EPOCHS}")
    model.learn(
        total_timesteps=TIMESTEPS, 
        reset_num_timesteps=False, 
        tb_log_name=experiment_name,
        callback=[eval_callback, episode_counter], 
        progress_bar=True
    )
    model.save(model_save_path)

c:\Users\Reuel Group\AppData\Local\Programs\Python\Python312\Lib\site-packages\rich\live.py:231: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

-------------------- Running Experiment: sac_cn10cb100_1_ret -------------------- 
-------------------- TRAINING SAC --------------------
Training 1/1


c:\Users\Reuel 
Group\AppData\Local\Programs\Python\Python312\Lib\site-packages\stable_baselines3\common\evaluation.py:67: 
UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting 
modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first
with ``Monitor`` wrapper.
  warnings.warn(

Eval num_timesteps=530000, episode_reward=10099.60 +/- 1591.91

Episode length: 99.76 +/- 15.74

...

Episode length: 102.04 +/- 16.77

Eval num_timesteps=620000, episode_reward=10812.20 +/- 1973.67

Episode length: 104.94 +/- 19.10

Stopping training because there was no new best model in the last 2 evaluations

Training stopped after 962 episodes.

In [12]:
print("Episode counts during training:")
print(episode_counts)

Episode counts during training:
[962]
